# Entity Ranking Evaluation: Headline Entity Recovery F1

This notebook evaluates the **EntiLytics entity ranking feature** using **Headline Entity Recovery F1** as the evaluation metric.

---

## Overview

EntiLytics uses a transformer-based entity ranker that scores named entities extracted from the article body using Manhattan distance between BERT embeddings. Entities with a distance score at or below the threshold of 0.45 are considered salient. This evaluation measures whether the top-ranked entities predicted by the system match the named entities that human editors considered important enough to include in the article headline.

---

## Gold Standard: Headline Entities as Automated Proxy

No benchmark dataset currently exists that pairs full news articles with human-ranked entity importance labels for this evaluation setup. As a result, article **headlines are used as an automated proxy** for central entity identification.

This approach is grounded in two supporting works:

**Singh et al. (2021)** demonstrate that named entities are among the most informationally significant elements of news headlines. Their study shows that preserving named entities is a primary concern when generating headlines from full articles, which supports the assumption that entities appearing in a headline are representative of the central entities in the corresponding article body.

**Baraniak and Sydow (2021)** further establish that named entities in headlines serve as the primary subjects of news content, forming the basis of their entity-level news analysis dataset.

### Important Limitation: NER-Dependent Gold Standard

The gold entities are extracted from headlines using the **same Flair NER model** that the EntiLytics system uses internally. This means the gold standard is not independently verified. Any systematic errors or biases in the Flair model will affect both the gold set and the system predictions. The headline-entity approach is an automated proxy for this configuration, but its precision as a gold standard cannot be fully guaranteed.

---

## Why a Lower Bound Score is Used

The Headline Entity Recovery F1 reported here should be interpreted as a **conservative lower-bound baseline**, not an exact measure of ranking quality. There are three systematic reasons this score underestimates real performance:

**1. Acronym and expansion mismatches.** Headlines frequently use abbreviations (e.g. `MMDA`, `DOJ`) while the article body contains their expanded forms (`Metropolitan Manila Development Authority`, `Department of Justice`). Even with partial acronym matching, not all abbreviation styles are captured, causing legitimate matches to be counted as misses.

**2. Valid entities not present in the headline.** The ranker correctly surfaces entities that are prominent in the article body but absent from the headline. Headlines are space-constrained and may name only the most recognizable entity for engagement purposes, omitting others that are equally or more prominent in the article content. These correct predictions reduce Precision without representing ranking errors.

**3. Editorial salience vs. content prominence mismatch.** Headlines prioritize reader recognition, while the ranker prioritizes semantic proximity to the article body. A headline may name a person for rhetorical effect while the article is primarily about an organization. The ranker correctly emphasizes the organization, but the headline-based gold standard does not reward this.

---

## Evaluation Metric

**Headline Entity Recovery F1** is used as the evaluation metric, consistent with the summarization evaluation. The same F1 formulation is applied:

$$\text{Precision} = \frac{|\text{headline entities found in top-K ranked}|}{|\text{top-K ranked entities}|}$$

$$\text{Recall} = \frac{|\text{headline entities found in top-K ranked}|}{|\text{total headline entities}|}$$

$$F1 = \frac{2 \times \text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$

Top-K is set per article equal to the number of gold headline entities, ensuring the comparison is fair.

---

## References

- Singh, B., Marathe, A., Rizvi, A. A., and Joshi, A. R. (2021). Retaining named entities for headline generation. In *Inventive Computation and Information Technologies*.
- Baraniak, K., Sydow, M. (2021). A dataset for Sentiment analysis of Entities in News headlines (SEN). Polish-Japanese Academy of Information Technology / Polish Academy of Sciences.

---

## Section 1: Article Collection

Articles are fetched from three RSS feeds: **Manila Times**, **Tech Pinas**, and **The Guardian**. All articles available from each feed are collected with no per-feed cap. Each article must pass three filters:

- Non-empty headline and body text
- Body text of at least 100 characters (excludes extremely short RSS descriptions)
- No pipe character `|` in the headline (a common sign of templated or aggregated titles)

The collected articles are saved to `summarization_dataset.csv`. This is the same dataset used by the summarization evaluation, ensuring both evaluations run on identical article sets for comparability.

In [ ]:
import pandas as pd
import sys
from pathlib import Path
from bs4 import BeautifulSoup

base_path = Path(".").resolve()
sys.path.append(str(base_path.parent))

from features.rss_handler import fetch_rss_articles

# RSS feed sources
FEEDS = {
    "manila_times" : "https://www.manilatimes.net/news/feed/",
    "tech_pinas"   : "http://feeds.feedburner.com/Techpinas",
    "the_guardian" : "https://www.theguardian.com/international/rss",
}


def clean(text):
    """Remove HTML tags and normalize whitespace using BeautifulSoup."""
    if not text:
        return ""
    soup = BeautifulSoup(text, "html.parser")
    return soup.get_text(separator=" ").strip()


# Collect articles from all feeds
collected_article_rows = []
current_article_id = 1

for source_name, feed_url in FEEDS.items():
    raw_articles_list = fetch_rss_articles(feed_url)
    articles_collected_from_source = 0

    for article_data in raw_articles_list:
        article_headline  = clean(article_data.get("title", ""))
        article_full_text = clean(article_data.get("description", ""))

        if not article_headline or not article_full_text:
            continue
        if len(article_full_text) < 100:
            continue
        if "|" in article_headline:
            continue

        collected_article_rows.append({
            "article_id" : current_article_id,
            "source"     : source_name,
            "headline"   : article_headline,
            "full_text"  : article_full_text,
        })
        current_article_id += 1
        articles_collected_from_source += 1

    print(f"{source_name}: collected {articles_collected_from_source} articles")

articles_dataframe = pd.DataFrame(collected_article_rows)
csv_output_path = base_path / "summarization_dataset.csv"
articles_dataframe.to_csv(csv_output_path, index=False)

print(f"\nTotal articles saved : {len(articles_dataframe)}")
print(f"Saved to             : {csv_output_path}")

---

## Section 2: Load and Filter Dataset

The saved CSV is loaded and stub articles are removed. Stub articles are RSS entries where the `full_text` field contains only a placeholder phrase such as "The post X appeared first on Y." with no actual article body. These provide no meaningful content for the ranker to work with and are excluded before evaluation.

In [ ]:
from features.flair_ner import identify_entities
from features.entity_ranking_and_summarization import entity_ranking

csv_dataset_path = base_path / "summarization_dataset.csv"
if not csv_dataset_path.exists():
    raise FileNotFoundError("summarization_dataset.csv not found. Run Section 1 first.")

articles_df = pd.read_csv(csv_dataset_path)
print(f"Loaded {len(articles_df)} articles")

# Remove RSS stub articles with no real body text
initial_count = len(articles_df)
articles_df = articles_df[~articles_df["full_text"].str.strip().str.match(r"^The post .+ appeared first on .+\.$")]
print(f"Removed {initial_count - len(articles_df)} stub articles, {len(articles_df)} remaining")

print()
print(articles_df.groupby("source").size().rename("article count").to_frame().to_string())

---

## Section 3: Gold Standard and Matching Functions

Named entities are extracted from each article headline using the Flair NER model. These headline entities serve as the gold standard for the evaluation.

A phrase filter (`is_valid_gold_entity`) removes Flair false positives — long noun phrases or clause fragments that headline grammar causes the NER model to incorrectly tag as named entities. Spans longer than 5 words or containing clause-marker verbs are rejected.

Entity matching uses a composite strategy to handle surface form differences between headline abbreviations and body-text expansions:
- **Exact match** - always accepted
- **Short token guard** - tokens under 3 characters only match via acronym to avoid false hits (e.g. `'us'` matching `'house'`)
- **Substring containment** - handles partial name matches (e.g. `'israel'` matching `'israeli'`)
- **Strict acronym** - first letters of all words (e.g. `'doj'` matching `'department of justice'`)
- **Partial acronym** - first letters of content words, excluding function words commonly dropped in acronym formation

In [ ]:
def is_acronym_of(short, long):
    """Check if short is an acronym of long."""
    if len(short) < 2:
        return False
    words = [w for w in long.split() if w]
    if len(words) < len(short):
        return False
    return short == "".join(w[0] for w in words)


def is_partial_acronym_of(short, long):
    """Check if short is an acronym ignoring function words."""
    if len(short) < 2:
        return False
    stop_words = {"for", "of", "the", "and", "in", "at", "by", "a", "an"}
    content_words = [w for w in long.split() if w not in stop_words]
    if len(content_words) < len(short):
        return False
    return short == "".join(w[0] for w in content_words)


def is_match(entity_a, entity_b):
    """Check if two entities are the same (handles exact, substring, and acronym variations)."""
    a = entity_a.strip().lower()
    b = entity_b.strip().lower()

    if not a or not b:
        return False
    if a == b:
        return True
    if len(a) < 3 or len(b) < 3:
        return (is_acronym_of(a, b) or is_acronym_of(b, a) or
                is_partial_acronym_of(a, b) or is_partial_acronym_of(b, a))
    if a in b or b in a:
        return True
    return (is_acronym_of(a, b) or is_acronym_of(b, a) or
            is_partial_acronym_of(a, b) or is_partial_acronym_of(b, a))


def is_valid_gold_entity(text):
    """Filter out NER mistakes - reject very long phrases and clauses that aren't true named entities."""
    words = text.split()
    if len(words) > 5:
        return False
    clause_markers = {"to", "would", "did", "have", "said", "says", "want", "wants"}
    if any(w in clause_markers for w in words):
        return False
    return True


def extract_gold_entities_from_headline(headline_text):
    """Get the named entities from headline, filtered to remove false positives."""
    entities = identify_entities(headline_text)
    return set(
        e["text"].lower() for e in entities
        if is_valid_gold_entity(e["text"].lower())
    )

---

## Section 4: Ranking Evaluation

For each article, the following steps are performed:

1. Gold entities are extracted from the headline using the function defined above.
2. Named entities are identified in the full article body using Flair NER.
3. `entity_ranking()` ranks those body entities by Manhattan distance to the article content.
4. The top-K ranked entities are selected, where K equals the number of gold headline entities.
5. Headline Entity Recovery F1 is computed by checking which gold entities match any of the top-K ranked predictions using the composite matcher.

Articles where the headline yields no valid named entities are skipped and counted separately.

In [ ]:
# Evaluation loop
ranking_precision_list = []
ranking_recall_list = []
ranking_f1_list = []
skipped_count = 0

print("ENTITY RANKING EVALUATION\n")

for _, article_row in articles_df.iterrows():
    # Get gold entities from headline
    gold_headline_entities = extract_gold_entities_from_headline(article_row["headline"])

    # Skip articles with no valid headline entities
    if not gold_headline_entities:
        skipped_count += 1
        continue

    # Rank entities from article body
    body_entities = identify_entities(article_row["full_text"])
    ranked_entities = entity_ranking(article_row["full_text"], body_entities)

    # Get top-K entities where K = number of gold entities (fair comparison)
    top_k = max(1, len(gold_headline_entities))
    system_top_entities = {e["name"].lower() for e in ranked_entities[:top_k]}

    # Check which system predictions match gold entities
    matched_entities = set(
        gold_e for gold_e in gold_headline_entities
        if any(is_match(gold_e, sys_e) for sys_e in system_top_entities)
    )

    # Calculate metrics
    tp = len(matched_entities)
    precision = tp / len(system_top_entities) if system_top_entities else 0
    recall = tp / len(gold_headline_entities) if gold_headline_entities else 0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0

    ranking_precision_list.append(precision)
    ranking_recall_list.append(recall)
    ranking_f1_list.append(f1)

    print(f"[{article_row['source']}] {article_row['headline'][:60]}...")
    print(f"Gold headline entities: {sorted(gold_headline_entities)}")
    print(f"Top-{top_k} ranked entities: {sorted(system_top_entities)}")
    print(f"Matched: {sorted(matched_entities)}")
    print(f"P={precision:.2f}  R={recall:.2f}  F1={f1:.2f}\n")

---

## Section 5: Results

Mean Precision, Recall, and F1 are computed across all evaluated articles, excluding skipped ones. Results are reported as a **conservative lower-bound baseline** for the reasons described in the introduction.

In [ ]:
# Print aggregate results
total_evaluated = len(ranking_f1_list)
if total_evaluated == 0:
    print("No articles evaluated.")
    sys.exit(0)

avg_precision = sum(ranking_precision_list) / total_evaluated
avg_recall = sum(ranking_recall_list) / total_evaluated
avg_f1 = sum(ranking_f1_list) / total_evaluated

print(f"RESULTS ({total_evaluated} articles evaluated, {skipped_count} skipped)\n")
print("ENTITY RANKING - Headline Entity Recovery F1")
print(f"Mean Precision: {avg_precision:.4f}")
print(f"Mean Recall: {avg_recall:.4f}")
print(f"Mean F1: {avg_f1:.4f}")
print(f"\n[Lower-bound baseline: Does not reward valid entities missing from headlines]")

---

## Section 6: Recorded Results and Interpretation

The results below were recorded from the evaluation run on **March 24, 2026**, using articles collected from Manila Times, Tech Pinas, and The Guardian.

| Metric | Value |
|---|---|
| Articles evaluated | 124 |
| Articles skipped (no headline entities) | 53 |
| Mean Precision | 0.4281 |
| Mean Recall | 0.3965 |
| Mean F1 | 0.4058 |

### Interpretation

**Recall (0.3965)** reflects the proportion of headline gold entities that the ranker successfully surfaces in its top-K predictions. The ranker scores entities by semantic proximity to the article body using Manhattan distance, which means it naturally prioritizes entities that dominate the article content. When a headline entity appears only briefly in the body, the ranker deprioritizes it in favor of entities with stronger textual presence, reducing recall.

**Precision (0.4281)** is higher than Recall, which is the opposite pattern from the summarization evaluation. This indicates the ranker is reasonably selective — when it does predict an entity, it is more likely to be correct than not. The precision gap reflects cases where the ranker surfaces legitimate content-prominent entities that the headline omitted for space or engagement reasons.

**F1 (0.4058)** is the harmonic mean of Precision and Recall. As a lower-bound baseline, the true F1 under a human-annotated gold standard would be expected to be higher, for the reasons described in the introduction. This result is consistent with the summarization evaluation's use of the same gold standard and the same systematic limitations.

### Sample Outputs

The examples below illustrate the most common outcome patterns observed across the three news sources.

**Tech Pinas - Full match:**
```
[tech_pinas] Infinix Celebrates Aurora Gaming PH M7 Victory with Infinix...
Gold headline entities: ['aurora gaming', 'infinix']
Top-2 ranked entities: ['aurora gaming ph', 'infinix honors aurora s rise to the top for infinix']
Matched: ['aurora gaming', 'infinix']
P=1.00  R=1.00  F1=1.00
```
Both gold entities matched via substring containment. The ranker correctly identified the two central entities despite their expanded forms in the body text.

**Tech Pinas - Partial match with acronym surface form:**
```
[tech_pinas] PLDT Enterprise and Lin Man Power Technology Roll Out Nation...
Gold headline entities: ['lin man power technology', 'pldt enterprise']
Top-2 ranked entities: ['lin man power technology', 'lin man power technology , inc']
Matched: ['lin man power technology']
P=0.50  R=0.50  F1=0.50
```
`PLDT Enterprise` was not surfaced in the top-2 because the ranker ranked the body-text expansion of Lin Man Power Technology twice. This is a ranking order issue rather than a matching failure.

**Tech Pinas - Surface form mismatch not caught by matching:**
```
[tech_pinas] Toyota Philippines' February 2026 Deals Will Make You Feel G...
Gold headline entities: ["toyota philippines '"]
Top-1 ranked entities: ['toyota motor philippines']
Matched: []
P=0.00  R=0.00  F1=0.00
```
The gold entity includes a trailing apostrophe from headline punctuation, and the body uses `Toyota Motor Philippines`. The apostrophe causes the gold string not to substring-match `toyota motor philippines`. This is a gold standard cleaning issue rather than a ranking failure.

**The Guardian - Editorial vs content prominence:**
```
[the_guardian] The Maga divide over Iran - podcast...
Gold headline entities: ['iran', 'maga']
Top-2 ranked entities: ['iran', 'maga']
Matched: ['iran', 'maga']
P=1.00  R=1.00  F1=1.00
```
When the headline and article body align on entity prominence, the ranker performs perfectly.